# Контрастная синтетика для проверки метрик чанкинга

Последняя ячейка выполняет по одному платному запросу для каждого файла из
`SELECTED_PROMPTS` и заменяет соответствующий JSON в `data/generated`.

In [9]:
from __future__ import annotations

import os
from pathlib import Path
import sys

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

MODEL_NAME = "deepseek-v4-pro"
BASE_URL = "https://api.deepseek.com"
TEMPERATURE = 1.0
MAX_TOKENS = 8192
TIMEOUT_SECONDS = 120.0

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "prompts").is_dir() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

from generation_pipeline import generate_json, save_json

PROMPTS_ROOT = PROJECT_ROOT / "prompts"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "generated"
SELECTED_PROMPTS = [
    Path("general_validation.md"),
    Path("metrics/size_compliance.md"),
    Path("metrics/intrachunk_cohesion.md"),
    Path("metrics/contextual_coherence.md"),
    Path("metrics/boundary_clarity.md"),
    Path("metrics/chunk_score.md"),
    Path("metrics/hope_concept_unity.md"),
    Path("metrics/hope_semantic_independence.md"),
    Path("metrics/hope_information_preservation.md"),
]

client = OpenAI(
    api_key=os.environ["API_KEY"],
    base_url=BASE_URL,
    timeout=TIMEOUT_SECONDS,
)


In [10]:
system_prompt = (PROMPTS_ROOT / "system.md").read_text(encoding="utf-8")

for prompt_path in SELECTED_PROMPTS:
    prompt_name = prompt_path.stem
    print(f"Generating {prompt_name}...")
    user_prompt = (PROMPTS_ROOT / prompt_path).read_text(encoding="utf-8")
    payload = generate_json(
        client,
        system_prompt,
        user_prompt,
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    output_path = save_json(payload, OUTPUT_ROOT / f"{prompt_name}.json")
    print(f"Saved {output_path}")


Generating general_validation...


JSONDecodeError: Expecting ',' delimiter: line 2 column 114 (char 115)